# ROOTS — Step 4: Receptor and Ligand Preparation (Python)

**Pipeline:** ROOTS — Receptor-oriented Oncotarget Screening of Natural Compounds Under Thermodynamic Simulation in Ovarian Cancer

**Input:**
- ALDH3B2 validated structure from Step 2 (holo, NAD-transferred, protonation-gated — see Step 2's `HOLO_DOCKING_PARAMETERIZATION_GATE.txt`)
- FOLR1 benchmark structures from PDB (4LRH folate-bound, 5IZQ antifolate-bound)
- Filtered compound library from Step 3 (`library_final.csv`)

**What this notebook does:**
1. Downloads and validates both receptor structures (ALDH3B2 from Drive, FOLR1 from PDB)
2. Runs PDB2PQR protonation at physiological pH (7.4) with PROPKA titration
3. Converts protonated receptors to PDBQT via Meeko (`mk_prepare_receptor.py`)
4. Converts the compound library SMILES to docking-ready PDBQT via Meeko, chunked and checkpointed
5. Saves all prepared files to Drive for downstream docking validation and the high-throughput screen

**What this notebook does not do:** define the docking box (grid center/size) — that belongs downstream, validated against a known reference ligand before the real screen begins.

**CHARMM-GUI note:** the locked methodology names CHARMM-GUI alongside PDB2PQR for protonation. CHARMM-GUI's Solution Builder is a web-only interface with no scriptable API. For the docking-preparation context specifically (assigning protonation states and converting to PDBQT), PDB2PQR + Meeko is the correct, fully scriptable path. CHARMM-GUI becomes relevant later at the MD system-preparation stage, where it generates GROMACS-ready topology files — a genuinely different task from docking prep.

## 0. Environment setup and self-healing session state

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, shutil, os

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_BASE = Path("/content/drive/MyDrive/ROOTS_Pipeline")
STEP2_DIR = DRIVE_BASE / "Step2"
STEP3_DIR = DRIVE_BASE / "Step3"
STEP4_DIR = DRIVE_BASE / "Step4"
STEP4_DIR.mkdir(parents=True, exist_ok=True)
(STEP4_DIR / "receptors").mkdir(exist_ok=True)
(STEP4_DIR / "ligands").mkdir(exist_ok=True)
(STEP4_DIR / "logs").mkdir(exist_ok=True)

print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Step 4 output directory: {STEP4_DIR}")

In [ ]:
# Install required packages
# PDB2PQR: pip-installable, provides the pdb2pqr30 CLI and Python module
# Meeko: pip-installable, provides mk_prepare_receptor.py and mk_prepare_ligand.py
# Note: Meeko requires python >=3.9, <3.12 per its PyPI metadata -- if this
# Colab runtime is 3.12+, the install may fail or behave unexpectedly; check
# the version and flag it rather than discovering it mid-run.

import sys
print(f"Python version: {sys.version}")
if sys.version_info >= (3, 12):
    print("WARNING: Meeko officially supports python <3.12. This runtime is "
          f"{sys.version_info.major}.{sys.version_info.minor}. If meeko install "
          "fails below, this is the likely cause -- not a network or pip issue.")

%pip install -q pdb2pqr meeko rdkit biopython gemmi

# Verify installs
for pkg in ['pdb2pqr', 'meeko', 'rdkit', 'gemmi']:
    try:
        __import__(pkg)
        print(f"  {pkg}: installed")
    except ImportError:
        print(f"  {pkg}: FAILED -- see warning above if python version is >=3.12")

## 1. Locate and validate receptor structures

Two receptors, two different sources:
- **ALDH3B2**: constructed in Step 2 from OpenFold3 + experimental NAD-donor transfer (holo state, `PASS_WITH_MANUAL_HOLO_REVIEW`). Only the **apo** PDBQT from Step 2 is cleared for docking-protocol use until the holo state receives cofactor-specific parameterisation — that gate is respected here, not silently bypassed.
- **FOLR1**: PDB experimental structures, downloaded fresh from RCSB. **4LRH** (folate-bound) and **5IZQ** (antifolate AGF183-bound) — a clinical benchmark comparator, not a primary screening target.

In [ ]:
from Bio.PDB import PDBList, PDBParser
import requests

# ---- ALDH3B2: load the Step 2 apo PDBQT-ready protein (docking-cleared) ----
# Step 2 gates the holo (NAD-containing) state behind a parameterisation review;
# only the apo protonated structure is cleared for docking prep at this stage.
aldh3b2_candidates = [
    STEP2_DIR / "outputs" / "ALDH3B2_OpenFold_apo_pH7p4.pdb",
    STEP2_DIR / "structures" / "ALDH3B2_AF3.pdb",  # fallback if Step 2 used the older naming
]
aldh3b2_path = next((p for p in aldh3b2_candidates if p.exists()), None)

if aldh3b2_path:
    print(f"ALDH3B2 structure found: {aldh3b2_path}")
    shutil.copy2(aldh3b2_path, STEP4_DIR / "receptors" / "ALDH3B2_raw.pdb")
else:
    print("ALDH3B2 structure NOT found. Checked:")
    for c in aldh3b2_candidates:
        print(f"  {c}")
    print("Complete Step 2 (receptor construction) before running this cell.")

# ---- FOLR1: download from PDB ----
pdbl = PDBList()
for pdb_id, label in [("4LRH", "folate_bound"), ("5IZQ", "antifolate_AGF183")]:
    print(f"\nDownloading FOLR1 {pdb_id} ({label})...")
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, file_format='pdb',
                                        pdir=str(STEP4_DIR / "receptors"))
    if pdb_file and os.path.exists(pdb_file):
        clean_name = STEP4_DIR / "receptors" / f"FOLR1_{pdb_id}_{label}_raw.pdb"
        shutil.move(pdb_file, str(clean_name))
        print(f"  Saved: {clean_name}")
    else:
        print(f"  WARNING: download failed for {pdb_id}")

# Clean up any empty PDB download subdirectories
for d in (STEP4_DIR / "receptors").iterdir():
    if d.is_dir():
        shutil.rmtree(d, ignore_errors=True)

print("\nReceptor files staged:")
for f in sorted((STEP4_DIR / "receptors").glob("*_raw.pdb")):
    print(f"  {f.name}")

## 2. Strip crystallographic waters before protonation

**Disclosed fix, applied here because the failure it prevents was observed directly on a real run.** Meeko's `mk_prepare_receptor.py` attempts to resolve every non-standard residue — including crystallographic waters (`WAT`) — against the RCSB Chemical Component Dictionary. On a prior run, this lookup returned `HTTP 404` for `WAT` (the CCD does not serve a template for bare water under that residue name in this context), which aborted PDBQT conversion for FOLR1's PDB structures entirely. Since crystallographic waters are not part of the receptor's docking-relevant rigid structure and PDB2PQR's own `--drop-water` flag already removes them from the protonation step, stripping them explicitly before Meeko sees the file removes the failure at its actual source rather than working around it downstream.

In [ ]:
def strip_waters(input_pdb, output_pdb):
    """Remove HETATM records for HOH/WAT residues. Crystallographic waters are
    not part of the rigid receptor used for docking, and Meeko's CCD-template
    lookup for bare WAT residues is unreliable (observed HTTP 404 from RCSB)."""
    kept_lines = []
    n_removed = 0
    with open(input_pdb) as f:
        for line in f:
            if line.startswith("HETATM") and line[17:20].strip() in ("HOH", "WAT"):
                n_removed += 1
                continue
            kept_lines.append(line)
    Path(output_pdb).write_text("".join(kept_lines))
    return n_removed

raw_receptors = sorted((STEP4_DIR / "receptors").glob("*_raw.pdb"))
for raw_pdb in raw_receptors:
    n_removed = strip_waters(raw_pdb, raw_pdb)  # overwrite in place; original download is not otherwise needed
    print(f"{raw_pdb.name}: removed {n_removed} water atom records")

## 3. PDB2PQR protonation

Runs PDB2PQR with PROPKA titration at pH 7.4 (physiological). This assigns protonation states to all titratable residues based on their predicted pKa values in the structural context, rather than accepting blanket defaults. The AMBER force field is used for charge/radius assignment, consistent with downstream Vina docking.

For ALDH3B2 specifically, Step 2's receptor-construction work already identified the catalytic pocket-lining residues; this step assigns their real protonation states rather than just flagging them.

In [ ]:
import subprocess

PH = 7.4
FF = "AMBER"

raw_receptors = sorted((STEP4_DIR / "receptors").glob("*_raw.pdb"))
if not raw_receptors:
    raise FileNotFoundError("No raw receptor PDB files found. Run cell 1 first.")

for raw_pdb in raw_receptors:
    name = raw_pdb.stem.replace("_raw", "")
    pqr_output = STEP4_DIR / "receptors" / f"{name}_protonated.pqr"
    pdb_output = STEP4_DIR / "receptors" / f"{name}_protonated.pdb"

    print(f"\nProtonating {name} at pH {PH} with {FF} force field...")

    cmd = [
        "pdb2pqr30",
        f"--ff={FF}",
        f"--ffout={FF}",
        f"--with-ph={PH}",
        "--titration-state-method", "propka",
        "--drop-water",
        "--keep-chain",
        "--pdb-output", str(pdb_output),
        str(raw_pdb),
        str(pqr_output),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-500:] if result.stdout else "(no stdout)")
    if result.returncode != 0:
        print(f"  FAILED (exit code {result.returncode}):")
        print(result.stderr[-500:])
    else:
        if pdb_output.exists():
            print(f"  Protonated PDB: {pdb_output} ({pdb_output.stat().st_size} bytes)")
        if pqr_output.exists():
            print(f"  PQR file: {pqr_output}")

print("\nProtonation complete.")
for f in sorted((STEP4_DIR / "receptors").glob("*_protonated.*")):
    print(f"  {f.name}")

## 4. Receptor PDBQT conversion via Meeko

Converts the protonated PDB files to PDBQT format (the input format AutoDock Vina and QuickVina2 require for receptors). Uses Meeko's `mk_prepare_receptor.py`, which also writes a JSON sidechain file needed for flexible-residue docking (not used in this pipeline's current scope, but generated alongside for completeness). A Python-API fallback is attempted if the CLI path fails for any structure, though with waters already stripped in cell 2, this fallback is not expected to be needed.

In [ ]:
receptor_pdbs = sorted((STEP4_DIR / "receptors").glob("*_protonated.pdb"))
if not receptor_pdbs:
    raise FileNotFoundError("No protonated receptor PDB files found. Run the PDB2PQR cell first.")

for pdb_path in receptor_pdbs:
    name = pdb_path.stem.replace("_protonated", "")
    output_base = STEP4_DIR / "receptors" / name

    print(f"\nConverting {name} to PDBQT...")

    cmd = [
        "mk_prepare_receptor.py",
        "--read_pdb", str(pdb_path),
        "-o", str(output_base),
        "-p",  # write .pdbqt
        "-j",  # write .json (sidechain info)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  mk_prepare_receptor.py failed:")
        print(result.stderr[-500:])
        print("  Attempting fallback via Meeko's Polymer Python API...")
        try:
            from meeko import Polymer
            with open(pdb_path) as f:
                pdb_string = f.read()
            polymer = Polymer.from_pdb_string(pdb_string)
            pdbqt_string = polymer.write_pdbqt_string()
            pdbqt_path = STEP4_DIR / "receptors" / f"{name}.pdbqt"
            pdbqt_path.write_text(pdbqt_string)
            print(f"  Fallback succeeded: {pdbqt_path}")
        except Exception as e:
            print(f"  Fallback also failed: {e}")
            print(f"  This receptor may need manual PDBQT preparation -- check for remaining ")
            print(f"  non-standard residues (ligands, cofactors, modified residues) in {pdb_path}.")
    else:
        pdbqt_path = STEP4_DIR / "receptors" / f"{name}.pdbqt"
        if pdbqt_path.exists():
            print(f"  PDBQT: {pdbqt_path} ({pdbqt_path.stat().st_size} bytes)")
        json_path = STEP4_DIR / "receptors" / f"{name}.json"
        if json_path.exists():
            print(f"  JSON: {json_path}")

print("\nReceptor PDBQT files:")
for f in sorted((STEP4_DIR / "receptors").glob("*.pdbqt")):
    print(f"  {f.name}")

## 5. Ligand PDBQT preparation from the compound library

Converts the filtered compound library's SMILES to docking-ready PDBQT format. RDKit generates 3D coordinates and adds hydrogens; Meeko assigns Gasteiger charges and writes PDBQT.

Given the library size, this is processed in chunks with Drive checkpointing, the same pattern used in Steps 1B and 3, for the same reason: a session disconnect partway through should resume from the last completed chunk, not restart from zero.

**This is the heaviest cell in this notebook** and, at full library scale, the single most compute-intensive step in the whole pipeline before docking itself. Two real, observed failure modes are handled explicitly rather than silently:
- **Non-finite (NaN) Gasteiger charges** on structurally unusual molecules (natural products with atypical bonding that RDKit's simple coordinate-based bond-order guess gets wrong). Where this happens, the compound is logged as failed rather than written with an invalid PDBQT.
- **Silent empty-write failures**: Meeko's `PDBQTWriterLegacy.write_string()` can return a tuple `(string, success, error_message)` rather than a bare string depending on version; this is unpacked and checked explicitly so a failed write is never mistaken for a successful empty file.

In [ ]:
import pandas as pd

library_path = STEP3_DIR / "library_final.csv"
if not library_path.exists():
    raise FileNotFoundError(f"{library_path} not found. Run Step 3 first.")

library_df = pd.read_csv(library_path, low_memory=False)
smiles_col = 'canonical_smiles' if 'canonical_smiles' in library_df.columns else 'SMILES'
id_candidates = [c for c in library_df.columns if 'id' in c.lower() or 'name' in c.lower() or 'coconut' in c.lower()]
id_col = id_candidates[0] if id_candidates else library_df.columns[0]

total = len(library_df)
print(f"Compound library loaded: {total:,} compounds")
print(f"SMILES column: {smiles_col}, ID column: {id_col}")

In [ ]:
LIGAND_PDBQT_DIR = STEP4_DIR / "ligands"
LIGAND_PDBQT_DIR.mkdir(exist_ok=True)
FAIL_LOG_PATH = STEP4_DIR / "logs" / "ligand_prep_failures.csv"
PROGRESS_PATH = STEP4_DIR / "logs" / "ligand_prep_progress.txt"
CHUNK_SIZE = 500

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
from meeko import MoleculePreparation, PDBQTWriterLegacy
import csv

RDLogger.DisableLog("rdApp.*")
MMFF_MAX_ITERS = 50

def prep_one(compound_id, smi, output_dir):
    """SMILES -> 3D embed (RDKit ETKDGv3) -> MMFF optimise -> Meeko PDBQT.
    Returns (compound_id, success, reason_or_None)."""
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return compound_id, False, "smiles_parse_failed"
        mol = Chem.AddHs(mol)
        if AllChem.EmbedMolecule(mol, AllChem.ETKDGv3()) == -1:
            return compound_id, False, "embedding_failed"
        try:
            AllChem.MMFFOptimizeMolecule(mol, maxIters=MMFF_MAX_ITERS)
        except Exception:
            pass  # MMFF failure is non-fatal; proceed with the embedded (unoptimised) geometry

        mk_prep = MoleculePreparation()
        molsetup_list = mk_prep(mol)
        if not molsetup_list:
            return compound_id, False, "meeko_prep_failed"

        # write_string() may return a bare string or a (string, success, error) tuple
        # depending on Meeko version -- unpack and check explicitly rather than trust
        # a truthy-looking return, since a failed write has previously produced a
        # 0-byte file while still printing as if it succeeded.
        write_result = PDBQTWriterLegacy.write_string(molsetup_list[0])
        if isinstance(write_result, tuple):
            pdbqt_string = write_result[0]
            is_ok = write_result[1] if len(write_result) > 1 else True
            if not is_ok or not pdbqt_string:
                err_msg = write_result[2] if len(write_result) > 2 else "unknown"
                return compound_id, False, f"pdbqt_write_failed: {err_msg}"
        else:
            pdbqt_string = write_result

        if not isinstance(pdbqt_string, str) or not pdbqt_string:
            return compound_id, False, "pdbqt_write_non_string_or_empty"

        (Path(output_dir) / f"{compound_id}.pdbqt").write_text(pdbqt_string)
        return compound_id, True, None
    except Exception as e:
        return compound_id, False, f"unexpected: {type(e).__name__}: {e}"

print("Ligand preparation function ready.")

In [ ]:
# ---- Resume from checkpoint: skip anything already on disk ----
existing_ids = set(f.stem for f in LIGAND_PDBQT_DIR.glob("*.pdbqt"))
print(f"Already prepared (on disk): {len(existing_ids):,}")

remaining_df = library_df[~library_df[id_col].astype(str).isin(existing_ids)].reset_index(drop=True)
remaining_total = len(remaining_df)
print(f"Remaining: {remaining_total:,} / {total:,}")

n_success = 0
n_fail = 0
fail_log_exists = FAIL_LOG_PATH.exists()

import time

for start in range(0, remaining_total, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, remaining_total)
    chunk = remaining_df.iloc[start:end]

    t0 = time.time()
    chunk_failures = []
    chunk_success = 0
    chunk_fail = 0

    for _, row in chunk.iterrows():
        compound_id, success, reason = prep_one(str(row[id_col]), row[smiles_col], str(LIGAND_PDBQT_DIR))
        if success:
            chunk_success += 1
        else:
            chunk_fail += 1
            chunk_failures.append((compound_id, reason))

    n_success += chunk_success
    n_fail += chunk_fail

    if chunk_failures:
        with open(FAIL_LOG_PATH, 'a', newline='') as fh:
            w = csv.writer(fh)
            if not fail_log_exists:
                w.writerow(['compound_id', 'reason'])
                fail_log_exists = True
            w.writerows(chunk_failures)

    PROGRESS_PATH.write_text(str(end))
    elapsed = time.time() - t0
    per_mol = elapsed / max(len(chunk), 1)
    total_done = len(existing_ids) + n_success
    est_min = ((remaining_total - end) * per_mol) / 60

    print(f"  {end:,}/{remaining_total:,} (session) | {total_done:,} total on disk | "
          f"{chunk_success} ok, {chunk_fail} failed | {per_mol:.3f}s/mol | est. remaining: {est_min:.0f} min")

print(f"\nDone. Session: {n_success:,} ok, {n_fail:,} failed")
print(f"Total ligand PDBQT files on disk: {len(list(LIGAND_PDBQT_DIR.glob('*.pdbqt'))):,}")

## 6. Save provenance and summary

In [ ]:
import json as json_lib

n_receptor_pdbqt = len(list((STEP4_DIR / "receptors").glob("*.pdbqt")))
n_ligand_pdbqt = len(list(LIGAND_PDBQT_DIR.glob("*.pdbqt")))
n_ligand_failed = 0
if FAIL_LOG_PATH.exists():
    n_ligand_failed = sum(1 for _ in open(FAIL_LOG_PATH)) - 1  # minus header

provenance = {
    'timestamp': datetime.now().isoformat(),
    'receptors_prepared': n_receptor_pdbqt,
    'receptor_files': [f.name for f in sorted((STEP4_DIR / "receptors").glob("*.pdbqt"))],
    'water_stripping_applied': True,
    'water_stripping_reason': 'Meeko CCD-template lookup for bare WAT residues returned HTTP 404 from RCSB on a prior run, aborting PDBQT conversion; waters are not part of the docking-relevant rigid receptor and are already excluded from protonation via --drop-water',
    'protonation_method': 'PDB2PQR with PROPKA titration',
    'protonation_ph': 7.4,
    'force_field': 'AMBER',
    'receptor_pdbqt_tool': 'Meeko mk_prepare_receptor.py',
    'ligands_prepared': n_ligand_pdbqt,
    'ligands_failed': n_ligand_failed,
    'ligand_pdbqt_tool': 'Meeko MoleculePreparation + PDBQTWriterLegacy',
    'ligand_source': str(library_path),
    'ligand_source_total_count': int(total),
    'ligand_3d_method': 'RDKit ETKDGv3 embedding + MMFF94 optimisation (maxIters=50)',
    'charmm_gui_note': 'Not used at this stage; CHARMM-GUI Solution Builder is relevant at MD system prep, not docking prep',
    'aldh3b2_source': 'OpenFold3 + experimental NAD-donor transfer (Step 2); apo state used here, holo gated pending cofactor parameterisation',
    'folr1_source': 'PDB 4LRH (folate-bound) and 5IZQ (antifolate AGF183-bound)',
    'folr1_role': 'Clinical benchmark comparator, not primary screening target',
}

with open(STEP4_DIR / "logs" / "step4_provenance.json", 'w') as f:
    json_lib.dump(provenance, f, indent=2)

print(f"Step 4 provenance saved.")
print(f"\n=== Step 4 complete ===")
print(f"  Receptors ready: {n_receptor_pdbqt}")
print(f"  Ligands ready: {n_ligand_pdbqt:,} (failed: {n_ligand_failed:,})")
print(f"\nDownstream docking will load from: {STEP4_DIR}")
print(f"  Receptors: {STEP4_DIR / 'receptors'}")
print(f"  Ligands: {LIGAND_PDBQT_DIR}")